In [0]:

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.meta_demo")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.meta_demo.raw")
base = "/Volumes/workspace/meta_demo/raw"

customers = spark.createDataFrame(
    [(1,"Ravi","Pune"),(2,"Anu","Delhi"),(3,"Kiran","Chennai")],
    "cust_id INT, name STRING, city STRING")
orders = spark.createDataFrame(
    [(101,1,500.0),(102,2,250.0),(103,1,120.0)],
    "order_id INT, cust_id INT, amount DOUBLE")
products = spark.createDataFrame(
    [(11,"Laptop"),(12,"Phone")],
    "prod_id INT, prod_name STRING")

customers.write.mode("overwrite").parquet(f"{base}/customers")
orders.write.mode("overwrite").option("header","true").csv(f"{base}/orders")
products.write.mode("overwrite").json(f"{base}/products")

display(dbutils.fs.ls(base))

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.meta_demo.pipeline_config (
  source_name  STRING,
  source_path  STRING,
  format       STRING,
  target_table STRING,
  load_type    STRING,   -- overwrite | append | merge
  primary_key  STRING,
  active       STRING    -- Y | N
)""")

spark.sql("""
INSERT INTO workspace.meta_demo.pipeline_config VALUES
 ('customers','/Volumes/workspace/meta_demo/raw/customers','parquet','workspace.meta_demo.silver_customers','merge','cust_id','Y'),
 ('orders',   '/Volumes/workspace/meta_demo/raw/orders',   'csv',    'workspace.meta_demo.silver_orders',   'append','order_id','Y'),
 ('products', '/Volumes/workspace/meta_demo/raw/products', 'json',   'workspace.meta_demo.silver_products', 'overwrite','prod_id','N')
""")

display(spark.table("workspace.meta_demo.pipeline_config"))

In [0]:
configs = spark.read.table("workspace.meta_demo.pipeline_config").filter("active = 'Y'").collect()
configs

In [0]:

from pyspark.sql.functions import current_timestamp, lit, input_file_name
from delta.tables import DeltaTable

# 1. Read the active configurations
configs = spark.read.table("workspace.meta_demo.pipeline_config").filter("active = 'Y'").collect() #It takes the data from the Spark cluster and brings it to the Driver Node as a Python list. You need it in Python so you can loop through it.

for row in configs:
    source_name = row["source_name"]
    source_path = row["source_path"]
    fmt         = row["format"]
    target_tbl  = row["target_table"]
    load_type   = row["load_type"]
    pk          = row["primary_key"]

    print(f"==================================================")
    print(f"Processing: {source_name} | Load Type: {load_type}")
    print(f"==================================================")

    # 2. READ (Dynamically read based on format)
    if fmt == "csv":
        df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(source_path)
    elif fmt == "json":
        df = spark.read.format("json").load(source_path)
    elif fmt == "parquet":
        df = spark.read.format("parquet").load(source_path)
    else:
        print(f"Unsupported format: {fmt}")
        continue

    # 3. ADD METADATA COLUMNS (Auditing)
    df = df.withColumn("_ingested_at", current_timestamp()) \
           .withColumn("_source_file", lit(source_path))

    # 4. WRITE (Dynamically apply load_type)
    if load_type == "overwrite":
        df.write.format("delta").mode("overwrite").saveAsTable(target_tbl)
        print(f"✅ OVERWRITE complete: {target_tbl}")

    elif load_type == "append":
        df.write.format("delta").mode("append").saveAsTable(target_tbl)
        print(f"✅ APPEND complete: {target_tbl}")

    elif load_type == "merge":
        # Check if target table exists. If not, create it using a normal write.
        if not spark.catalog.tableExists(target_tbl):
            df.write.format("delta").mode("overwrite").saveAsTable(target_tbl)
            print(f"✅ MERGE target created (initial load): {target_tbl}")
        else:
            # Perform Delta MERGE (Upsert)
            delta_table = DeltaTable.forName(spark, target_tbl)
            
            # Dynamically build the merge condition
            merge_condition = f"target.{pk} = source.{pk}"
            
            (delta_table.alias("target")
                .merge(df.alias("source"), merge_condition)
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute())
            print(f"✅ MERGE (Upsert) complete: {target_tbl}")

print("\n🎉 All active pipelines processed successfully.")

A Metadata-Driven Framework is the holy grail of Data Engineering. It is the exact pattern that companies use to avoid writing 100 different pipeline scripts for 100 different tables.

Instead of hardcoding table names, column names, and logic, you build one generic pipeline that reads a configuration table (metadata) and executes the logic dynamically.

Since you have an interview coming up, mastering this concept will make you look like a Senior Engineer. Let's build a simplified version of this in Databricks.

We will build a framework that ingests data from AWS S3 (using Auto Loader) into Databricks, but instead of hardcoding the source path and target table, it reads those instructions from a Control Table.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.pipeline_config (
    pipeline_id STRING,
    source_path STRING,
    target_table STRING,
    file_format STRING,
    load_type STRING,      -- 'FULL' or 'INCREMENTAL'
    is_active BOOLEAN
);

-- Insert metadata for 3 tables we want to ingest
INSERT INTO workspace.default.pipeline_config VALUES
('github_repos', 's3://my-bucket/raw/github/repos/', 'workspace.default.bronze_github_repos', 'json', 'INCREMENTAL', true),
('sales_data', 's3://my-bucket/raw/sales/', 'workspace.default.bronze_sales', 'csv', 'INCREMENTAL', true),
('customer_data', 's3://my-bucket/raw/customers/', 'workspace.default.bronze_customers', 'parquet', 'FULL', false);

In [0]:
%python
from pyspark.sql.functions import current_timestamp, input_file_name

# 1. Read the metadata table to get active pipelines
config_df = spark.read.table("workspace.default.pipeline_config") \
    .filter("is_active = true") \
    .collect()

# 2. Loop through each configuration row
for row in config_df:
    pipeline_id = row["pipeline_id"]
    source_path = row["source_path"]
    target_table = row["target_table"]
    file_format = row["file_format"]
    
    print(f"Starting ingestion for: {pipeline_id}")

    # 3. Dynamically build the Auto Loader stream
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.schemaLocation", f"/checkpoints/{pipeline_id}/schema")
        .load(source_path)
        # Add ingestion metadata for auditing
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", input_file_name())
    )

    # 4. Write the stream to the dynamic target table
    (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", f"/checkpoints/{pipeline_id}/checkpoint")
        .trigger(availableNow=True) # Process new files and stop
        .toTable(target_table)
    )
    
    print(f"Successfully loaded {pipeline_id} to {target_table}")

print("Metadata-driven ingestion complete.")

"What if the source schema changes?"
You answer: "Because I am using Auto Loader with a unique schemaLocation for each pipeline (/checkpoints/{pipeline_id}/schema), the framework will automatically evolve the schema if a new column is added to the source."

```
.saveAsTable(target_tbl) is used for Batch writes (Standard DataFrames).

.toTable(target_tbl) is used for Streaming writes (Streaming DataFrames).
```